In [69]:
import torch
from torch.utils.data import Dataset, DataLoader

import numpy as np
from typing import Tuple

Distinction between a Dataset and Dataloader
1. Dataset -> returns single sample
2. Dataloader -> builds batches from Dataset

__getitem__() returns samples. conversion from samples to batches is via the collate_fn() which is contained within the DataLoader function. collate_fn() takes tuples/dicts & converts it into batched tensors.

## Phase 1

Create a dataset that returns x = int and y = x^2, store numbers internally, implement __len__ and implement __getitem__.

In [28]:
def create_synthetic_data(input_samples: int=10000) -> np.ndarray:
    """
    Creates a synthetic dataset where the input is an integer/range of integers and the output squares that sum.
    Args:
        input_samples: number of samples to generate within the given input range.
    Returns:
        2D numpy array of input value and it's squared value in the form [input_value, squared_value].
    """
    x = np.arange(0,input_samples)
    y = x**2
    return np.stack((x,y), axis=1)

print("Loading synthetic data...")
synthetic_data = create_synthetic_data(input_samples=10000)
print(f"Data shape: {synthetic_data.shape}")
print(f"Print first ten elements as a brief overview of the dataset: {synthetic_data[1:10]}")
print(f"Test indexing for __getitem()__ function: {synthetic_data[6,0]}")

Loading synthetic data...
Data shape: (10000, 2)
Print first ten elements as a brief overview of the dataset: [[ 1  1]
 [ 2  4]
 [ 3  9]
 [ 4 16]
 [ 5 25]
 [ 6 36]
 [ 7 49]
 [ 8 64]
 [ 9 81]]
Test indexing for __getitem()__ function: 6


In [67]:
class MyCustomDataSet(Dataset):
    """
    Read and load the synthetic dataset.
    """
    def __init__(self, dataset):
        self.data = dataset
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        DataLoader calls this method to get individual samples.
        It automatically handles batching, shuffling, etc.

        Returns:
            input_sample: the input sample
            output_sample: the squared sample
        """
        # extract input samples
        input_sample = self.data[idx][0]
        output_sample = self.data[idx][1]

        # important: convert to PyTorch tensors
        # DataLoader expects tensors, not numpy arrays
        return torch.tensor(input_sample), torch.tensor(output_sample)

In [68]:
dataset = MyCustomDataSet(dataset=synthetic_data)
len(dataset)
dataset[6]

(tensor(6), tensor(36))

## Phase 2

Implement a DataLoader wrapper around the generated synthetic class. 

PyTorch concept: DataLoader

DataLoader wraps the Dataset and provides:
- Automatic batching (groups samples together)
- Shuffling (randomizes order for training)
- Parallel loading (speeds up data loading)
- Memory management (loads data as needed)

In [91]:
print("Creating DataLoader:")

standard_loader = DataLoader(
    dataset, # our custom dataset
    batch_size = 10, # 10 samples per batch
    shuffle=True, # randomize order, good for training
    num_workers=0,
    pin_memory=False
)

print(f"DataLoader created with {len(standard_loader)} batches.")
print("\nInspecting first batch from the DataLoader:")
for batch_idx, (batch_input, batch_output )in enumerate(standard_loader):
    print(f"Batch {batch_idx + 1}:")
    print(f"    Input shape: {batch_input.shape}")
    print(f"    Output shape: {batch_output.shape}")
    print(f"    Input batch value: {batch_input}")
    print(f"    Output batch value: {batch_output}")
    print(f"    Input data type: {batch_input.dtype}")
    print(f"    Output data type: {batch_output.dtype}")

    # show first sample in the batch
    print(f"\n  First sample input range: [{batch_input.min()},{batch_input.max()}]")
    print(f"\n  First sample output range: [{batch_output.min()}, {batch_output.max()}]")
    break

Creating DataLoader:
DataLoader created with 1000 batches.

Inspecting first batch from the DataLoader:
Batch 1:
    Input shape: torch.Size([10])
    Output shape: torch.Size([10])
    Input batch value: tensor([2914, 2049,  537, 7480, 4442, 7564, 6479, 2243, 1591, 1273])
    Output batch value: tensor([ 8491396,  4198401,   288369, 55950400, 19731364, 57214096, 41977441,
         5031049,  2531281,  1620529])
    Input data type: torch.int64
    Output data type: torch.int64

  First sample input range: [537,7564]

  First sample output range: [288369, 57214096]
